# Entrenamiento LoRA (Low-Rank Adaptation) sobre Stable Diffusion 1.5
## Máster en Inteligencia Artificial, Cloud Computing y DevOps
## Módulo: Generacion de Imagenes con AI
## Autor: Alexander De Sousa

Este notebook presenta el proceso de entrenamiento mediante LoRA (Low-Rank Adaptation) aplicado a Stable Diffusion 1.5 para aprender el estilo visual de ilustraciones y portadas de libros antiguos.

A diferencia del finetuning completo, LoRA permite entrenar únicamente un conjunto reducido de adaptadores insertados en las capas de atención de la UNet, lo que reduce drásticamente:

- el tiempo de entrenamiento,
- el consumo de memoria,
- y el tamaño final del modelo.

El objetivo es obtener un modelo ligero y eficiente que capture el estilo del dataset, manteniendo la calidad visual y permitiendo su reutilización en cualquier pipeline compatible.
---


#### 1. Instalación de dependencias
Instala todas las librerías necesarias para entrenar y aplicar LoRA sobre Stable Diffusion.

In [ ]:
!pip install -q "diffusers[torch]" "datasets>=2.16,<2.17" accelerate transformers pillow huggingface_hub

#### 2. Imports y configuración del dispositivo
Carga las clases principales.

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from datasets import load_dataset
from diffusers import (
    StableDiffusionPipeline,
    UNet2DConditionModel,
    AutoencoderKL,
    DDPMScheduler,
    LoraConfig
)
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
from huggingface_hub import upload_folder


 *Detecta si se usará GPU o CPU para el entrenamiento.*

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo detectado:", device)
if device == "cuda":
    !nvidia-smi


#### 3. Configuración base y carga del dataset
Define el modelo preentrenado, carga el dataset y selecciona un subconjunto manejable para el finetuning.

In [ ]:
pretrained_model_name = "CompVis/stable-diffusion-v1-4"
dataset_name = "gigant/oldbookillustrations"
max_train_samples = 200

dataset = load_dataset(dataset_name, split="train")
if max_train_samples:
    dataset = dataset.select(range(max_train_samples))

print("Claves del dataset:", dataset[0].keys())


#### 4. Transformaciones e inicialización de tokenizer
Prepara las imágenes a 512×512 y convierte los textos en tokens que el modelo puede procesar.

In [ ]:
resolution = 512

image_transforms = transforms.Compose([
    transforms.Resize(resolution),
    transforms.CenterCrop((resolution, resolution)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name, subfolder="tokenizer")


#### 5. Dataset wrapper (igual que en el principal)
Crea un dataset compatible con PyTorch que devuelve imágenes transformadas y textos tokenizados.

In [ ]:
class OldBookDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, image_transforms):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.image_transforms = image_transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]

        image: Image.Image = example["1600px"].convert("RGB")
        image = self.image_transforms(image)

        text = example["info_alt"]
        tokens = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )

        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids.squeeze(0),
            "attention_mask": tokens.attention_mask.squeeze(0),
        }

train_dataset = OldBookDataset(dataset, tokenizer, image_transforms)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)
len(train_dataset)


#### 6. Carga de los componentes del modelo base
Carga el VAE, el text encoder y la UNet del modelo Stable Diffusion original.

In [ ]:
if device == "cuda":
    torch.cuda.empty_cache()

noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name, subfolder="scheduler")
text_encoder = CLIPTextModel.from_pretrained(pretrained_model_name, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(pretrained_model_name, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(pretrained_model_name, subfolder="unet").to(device)


#### 7. Congelar VAE y text encoder
Evita entrenar módulos innecesarios para que solo se ajusten los pesos LoRA de la UNet.

In [ ]:
vae.eval()
text_encoder.eval()

for p in vae.parameters():
    p.requires_grad = False

for p in text_encoder.parameters():
    p.requires_grad = False


#### 8. Configurar LoRA Rank 4 en la UNet
Añadimos adaptadores LoRA (r=4) a los módulos de atención de la UNet, permitiendo aprender del dataset mediante matrices de bajo rango sin alterar los pesos base.
*El rank determina la capacidad del LoRA: Rank 4 es ligero y suficiente para aprender estilos, mientras que ranks más altos aumentan la calidad pero también el tamaño y el coste del entrenamiento.*

In [ ]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.0,
    bias="none"
)

unet.add_adapter(lora_config)
print("Adaptador LoRA Rank 4 añadido a la UNet.")

#### 9. Preparar entrenamiento LoRA
Selecciona únicamente los parámetros LoRA entrenables y configura el optimizador y Accelerator.  
Solo entrenamos los parámetros LoRA (los demás siguen congelados)

In [ ]:
learning_rate = 1e-4
num_epochs = 2

lora_params = [p for p in unet.parameters() if p.requires_grad]
print("Parámetros entrenables (LoRA):", sum(p.numel() for p in lora_params))

optimizer = torch.optim.AdamW(lora_params, lr=learning_rate)

accelerator = Accelerator()
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)
noise_scheduler.to(accelerator.device)
text_encoder.to(accelerator.device)
vae.to(accelerator.device)

#### 10. Bucle de entrenamiento (LoRA)
Ejecuta el proceso de aprendizaje donde la UNet ajusta solo los adaptadores LoRA para aprender el estilo del dataset.

In [ ]:
unet.train()

for epoch in range(num_epochs):
    progress_bar = tqdm(train_dataloader, desc=f"Epoch LoRA {epoch}", leave=True)

    for batch in progress_bar:
        with torch.no_grad():
            pixel_values = batch["pixel_values"].to(accelerator.device)
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * 0.18215

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=latents.device,
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        input_ids = batch["input_ids"].to(accelerator.device)
        encoder_hidden_states = text_encoder(input_ids)[0]

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

        loss = torch.nn.functional.mse_loss(noise_pred, noise)

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        progress_bar.set_postfix(loss=loss.item())


#### 11. Guardar solo los pesos LoRA
Guarda únicamente los adaptadores LoRA entrenados, generando un modelo pequeño y eficiente.

In [ ]:
lora_save_path = "./sd-oldbook-lora-r4"
os.makedirs(lora_save_path, exist_ok=True)

unet.save_pretrained(lora_save_path)
print("Pesos LoRA Rank 4 guardados en:", lora_save_path)


#### 12. Subir LoRA a Hugging Face
Subimos la carpeta del modelo finetuneado a nuestro repositorio personal, ubicado en *https://huggingface.co/*

In [ ]:
lora_repo_id = "alexdesousa/sd-oldbook-lora-r4"

upload_folder(
    folder_path=lora_save_path,
    repo_id=lora_repo_id,
    repo_type="model"
)

print("LoRA Rank 4 subido correctamente a Hugging Face:", lora_repo_id)


#### 13. Usar el LoRA para generar imágenes (comparativa)

***Antes de aplicar LoRA***  
Generamos una imagen con el modelo original.

In [ ]:
pipe_base = StableDiffusionPipeline.from_pretrained(
    pretrained_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

prompt = "an illustration of a medieval knight reading a book"

image_base = pipe_base(prompt).images[0]
image_base.save("modelo_base_lora.png")
image_base

***Despues de aplicar LoRA***  
Generamos la misma imagen con el modelo aplicando los pesos de LoRA (Rank 4)

In [ ]:
pipe_lora = StableDiffusionPipeline.from_pretrained(
    pretrained_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

pipe_lora.unet.load_attn_procs(lora_save_path)

image_lora = pipe_lora(prompt).images[0]
image_lora.save("modelo_lora.png")
image_lora